In [54]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML 


dt=1e-6 ; m=1; a=2 ; b=3 ; g=9.8 ; Gamm=3 ; w=4
tf=15; N=int(tf/dt);

t=np.linspace(0,tf,N)


x0= 5             #condiciones iniciales
v0= 0

## FUNCION PARA EL MOVIMIENTO


In [43]:
def Motion(a, b, Gamma, w, g, x0, v0, t):
    N = len(t)
    pos_x = np.zeros(N)
    v_x = np.zeros(N)
    a_x = np.zeros(N)

    # Condiciones iniciales
    pos_x[0] = x0
    v_x[0] = v0
    a_x[0] = a_num(pos_x[0], v_x[0], w, g, a, b, Gamma)

    for i in range(N - 1):
        dt = t[i+1] - t[i]

        # --- PREDICTOR (Euler) ---
        a_pred = a_num(pos_x[i], v_x[i], w, g, a, b, Gamma)
        v_pred = v_x[i] + a_pred * dt
        x_pred = pos_x[i] + v_x[i] * dt

        # --- CORRECTOR (Heun) ---
        pos_x[i+1] = pos_x[i] + 0.5 * (v_x[i] + v_pred) * dt
        
        a_next = a_num(x_pred, v_pred, w, g, a, b, Gamma)
        v_x[i+1] = v_x[i] + 0.5 * (a_pred + a_next) * dt
        
        
    # Posición z(x)
    pos_z = b * (pos_x / a)**Gamma
    a_x = a_num(pos_x, v_x, w, g, a, b, Gamma)
    
    return pos_x, pos_z, v_x, a_x

In [45]:
x , x_dot = sp.symbols('x x_dot' , real=True)
m_, a_ , b_, w_ , g_,Gamm_= sp.symbols('m a b w g Gamma', positive=True)

z=b_*(x/a_)**Gamm_
dzdx_sym= sp.diff(z, x)
dzdx2_sym= sp.diff(dzdx_sym, x)

a_sym = -(1 / (1 + dzdx_sym**2)) * (x_dot**2 * dzdx_sym * dzdx2_sym - x * w_**2 + g_ * dzdx_sym)
a_num = sp.lambdify((x, x_dot, w_, g_, a_, b_, Gamm_), a_sym, 'numpy')


In [46]:
#Simulación
x_sol, z_sol, v_sol, a_sol = Motion(a, b, Gamm, w, g, x0, v0, t)

## Lagrangiano, Hamiltoniano y Energía

In [ ]:
#Lagrangiano, Hamiltoniano y Energía
T_sym = sp.Rational(1,2)*m_*(x_dot**2 + w_**2*x**2 + (dzdx_sym*x_dot)**2) #para no meter float en T
L_sym = T_sym - m_*g_*z
E_sym = T_sym + m_*g_*z

p_sym = sp.diff(L_sym, x_dot)                  # derivada parcial
H_sym = sp.simplify(p_sym*x_dot - L_sym)      

args = (x, x_dot, m_, a_, b_, w_, g_, Gamm_)
L_num, p_num, H_num, E_num = (sp.lambdify(args, f, 'numpy') for f in (L_sym, p_sym, H_sym, E_sym))

pars = (m, a, b, w, g, Gamm)
L = L_num(x_sol, v_sol, *pars)
p = p_num(x_sol, v_sol, *pars)
H = H_num(x_sol, v_sol, *pars)
E = E_num(x_sol, v_sol, *pars)

print('max |E - H - m w² x²| =', np.abs(E - H - m*w**2*x_sol**2).max())

In [ ]:
#Equilibrio: x_dot = 0, x_ddot = 0  ->  w² x = g z'(x)
x_eq_sym = a_*(w_**2*a_**2/(Gamm_*g_*b_))**(1/(Gamm_-2))     # apartado (e)
x_eq_num = sp.lambdify((a_, b_, w_, g_, Gamm_), x_eq_sym, 'numpy')
x_eq = x_eq_num(a, b, w, g, Gamm)

# Comprobación: la aceleración en x_eq debe ser 0
print(f'x_eq = {x_eq:.6f} m,  a(x_eq) = {a_num(x_eq, 0, w, g, a, b, Gamm):.2e}')

# Estabilidad: Omega² = w²(γ-2)/(1+z'²)  (estable si γ>2, inestable si 1<γ<2)
zp_eq  = dzdx_sym.subs({x: x_eq, a_: a, b_: b, Gamm_: Gamm})
Omega  = np.sqrt(float(w**2*(Gamm-2)/(1 + zp_eq**2)))
print(f'Omega = {Omega:.4f} rad/s,  T = {2*np.pi/Omega:.4f} s')

In [ ]:
#Plot Lagrangiano, Hamiltoniano y Energía
def plot_LHE(t, L, H, E, s=None):
    s = s or max(1, len(t)//20000)                 # submuestreo
    fig, ax = plt.subplots(3, 1, figsize=(9, 8), sharex=True)

    ax[0].plot(t[::s], L[::s], color='tab:purple')
    ax[0].set(ylabel='L (J)', title='Lagrangiano')

    ax[1].plot(t[::s], E[::s], color='tab:blue', lw=2, label='E = T + V')
    ax[1].plot(t[::s], H[::s], 'r--', lw=1.5, label='H = p ẋ − L')
    ax[1].set(ylabel='J', title='E no se conserva, H sí'); ax[1].legend()

    ax[2].semilogy(t[::s], np.abs(H[::s]/H[0] - 1) + 1e-16, color='tab:red')
    ax[2].set(xlabel='t (s)', ylabel='|H/H₀ − 1|', title='Error numérico en H')

    for A in ax: A.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

plot_LHE(t, L, H, E)

In [ ]:
#Animación
plt.rcParams['animation.embed_limit'] = 60         # MB; el límite por defecto (20) se queda corto

def animate_bead(t, x_sol, z_sol, w, x_eq=None, fps=20, speed=1.5, trail_s=2.0):
    dt = t[1] - t[0]
    step = max(1, round(speed/(fps*dt)))
    idx = np.arange(0, len(t), step)
    xs, zs, ts = x_sol[idx], z_sol[idx], t[idx]
    ph = w*ts
    X, Y = xs*np.cos(ph), xs*np.sin(ph)            # posición inercial
    trail_n = int(trail_s/(dt*step))

    xw = np.linspace(0, 1.1*x_sol.max(), 300)
    zw = b*(xw/a)**Gamm
    R = xw[-1]

    fig, (A1, A2) = plt.subplots(1, 2, figsize=(11, 5))
    A1.plot(xw, zw, 'k-', lw=2)
    if x_eq is not None:
        A1.axvline(x_eq, color='g', ls=':', label='x_eq'); A1.legend()
    A1.set(xlabel='x (m)', ylabel='z (m)', title='Alambre (sistema giratorio)'); A1.grid(alpha=0.3)
    A2.set(xlim=(-R, R), ylim=(-R, R), aspect='equal', xlabel='X (m)', ylabel='Y (m)',
           title='Vista cenital (inercial)'); A2.grid(alpha=0.3)

    bead1, = A1.plot([], [], 'ro', ms=9)
    bead2, = A2.plot([], [], 'ro', ms=8)
    trail, = A2.plot([], [], 'r-', lw=1, alpha=0.5)
    wire,  = A2.plot([], [], 'k-', lw=2)
    txt = A1.text(0.03, 0.92, '', transform=A1.transAxes)

    def update(k):
        bead1.set_data([xs[k]], [zs[k]])
        bead2.set_data([X[k]], [Y[k]])
        j = max(0, k - trail_n); trail.set_data(X[j:k+1], Y[j:k+1])
        wire.set_data([0, R*np.cos(ph[k])], [0, R*np.sin(ph[k])])
        txt.set_text(f't = {ts[k]:.2f} s')
        return bead1, bead2, trail, wire, txt

    anim = animation.FuncAnimation(fig, update, frames=len(idx), interval=1000/fps, blit=True)
    plt.close(fig)
    return anim

anim = animate_bead(t, x_sol, z_sol, w, x_eq)
HTML(anim.to_jshtml())